##### Análise dados ENEM 2023

###### FILES

1. amostra - 9999 registros
2. MICRODADOS_ENEM_2023 - 3933955 registros

In [6]:
import pandas as pd
import difflib

csv_path = "DADOS/MICRODADOS_ENEM_2023.csv"
chunk_size = 500_000  # ajustar conforme sua RAM

target_col_normalized = "NU_INSCRICAO"

# Primeiro carregamos apenas o cabeçalho para ver os nomes das colunas
cols = pd.read_csv(csv_path, encoding="latin1", nrows=0).columns
print("Colunas (primeiras 40):", cols[:40].tolist())

# Normaliza o nome para evitar problemas de espaços/case
col_map = {c.strip().upper(): c for c in cols}
if target_col_normalized not in col_map:
    sugestoes = difflib.get_close_matches(target_col_normalized, col_map.keys(), n=10, cutoff=0.4)
    raise KeyError(
        f"Coluna {target_col_normalized!r} não encontrada no CSV. \n"
        f"Tente uma destas colunas similares (ou atualize target_col_normalized):\n"
        f"{sugestoes}"
    )
target_col = col_map[target_col_normalized]

acc = None
for chunk in pd.read_csv(csv_path, encoding="latin1", chunksize=chunk_size, usecols=[target_col]):
    counts = chunk[target_col].value_counts()
    acc = counts if acc is None else acc.add(counts, fill_value=0)

acc = acc.sort_values(ascending=False)
print(acc.head(20))

Colunas (primeiras 20): ['NU_INSCRICAO;NU_ANO;TP_FAIXA_ETARIA;TP_SEXO;TP_ESTADO_CIVIL;TP_COR_RACA;TP_NACIONALIDADE;TP_ST_CONCLUSAO;TP_ANO_CONCLUIU;TP_ESCOLA;TP_ENSINO;IN_TREINEIRO;CO_MUNICIPIO_ESC;NO_MUNICIPIO_ESC;CO_UF_ESC;SG_UF_ESC;TP_DEPENDENCIA_ADM_ESC;TP_LOCALIZACAO_ESC;TP_SIT_FUNC_ESC;CO_MUNICIPIO_PROVA;NO_MUNICIPIO_PROVA;CO_UF_PROVA;SG_UF_PROVA;TP_PRESENCA_CN;TP_PRESENCA_CH;TP_PRESENCA_LC;TP_PRESENCA_MT;CO_PROVA_CN;CO_PROVA_CH;CO_PROVA_LC;CO_PROVA_MT;NU_NOTA_CN;NU_NOTA_CH;NU_NOTA_LC;NU_NOTA_MT;TX_RESPOSTAS_CN;TX_RESPOSTAS_CH;TX_RESPOSTAS_LC;TX_RESPOSTAS_MT;TP_LINGUA;TX_GABARITO_CN;TX_GABARITO_CH;TX_GABARITO_LC;TX_GABARITO_MT;TP_STATUS_REDACAO;NU_NOTA_COMP1;NU_NOTA_COMP2;NU_NOTA_COMP3;NU_NOTA_COMP4;NU_NOTA_COMP5;NU_NOTA_REDACAO;Q001;Q002;Q003;Q004;Q005;Q006;Q007;Q008;Q009;Q010;Q011;Q012;Q013;Q014;Q015;Q016;Q017;Q018;Q019;Q020;Q021;Q022;Q023;Q024;Q025']


KeyError: "Coluna 'NU_INSCRICAO' não encontrada no CSV"

In [ ]:
import dask.dataframe as dd

ddf = dd.read_csv("seu_arquivo.csv", assume_missing=True)

# exemplo de análise
result = ddf["coluna_interessante"].value_counts().compute()

In [ ]:
import polars as pl

df = pl.scan_csv("seu_arquivo.csv")  # leitura lazy
result = df.groupby("coluna").agg(pl.count()).collect()